### **Parte 4: Transformación avanzada**

Importamos las librerias, le indicamos a python donde encontrar el archivo utils.py y cargamos el dataset que se limpio en 03_limpieza

In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Permitir que el cuaderno lea las funciones de la carpeta src
sys.path.append(os.path.abspath('../src'))
from utils import limpiar_generos, asignar_temporada

# Cargar los datos limpios y auditados 
ruta_limpia = '../data/processed/movies_unificado.csv'
df_transformado = pd.read_csv(ruta_limpia)

print(f"Dataset cargado con {df_transformado.shape[0]} filas y {df_transformado.shape[1]} columnas.")

Dataset cargado con 3229 filas y 20 columnas.


Tomamos la columna de texto release_date la convertimos a un formato de tiempo real (datatime) y extraemos atributos clave: el mes y el dia de la semana. luego, aplicamos la funcion modular para crear la columna "Temporada". esto transforma a un dato crudo en variables de altisimo valor para el modelo predictivo

In [2]:
# Convertir texto a formato datetime de Pandas
df_transformado['release_date'] = pd.to_datetime(df_transformado['release_date'])

# Extracción rápida de características
df_transformado['mes_estreno'] = df_transformado['release_date'].dt.month
df_transformado['dia_semana_estreno'] = df_transformado['release_date'].dt.dayofweek # 0=Lunes, 6=Domingo

# Aplicar la función modular de temporadas
df_transformado['temporada'] = df_transformado['mes_estreno'].apply(asignar_temporada)

print("Atributos temporales (Mes, Día, Temporada) extraídos exitosamente.")

Atributos temporales (Mes, Día, Temporada) extraídos exitosamente.


Primero, se aplana el json de géneros. Luego, utilizando el análisis de 01_eda, se crean variables numericas binarias

**Los modelos de Machine Learning no entienden palabras, solo números, por lo que este One-Hot Encoding manual es indispensable.**

In [18]:
# Celda 3: Aplanado de JSON, Encoding y Nuevas Variables (Directores y Estudios)

import pandas as pd
# Importamos LAS DOS funciones desde el archivo de tu equipo
from utils import extraer_director, limpiar_estudios

# --- 1. RECUPERAR AL DIRECTOR DESDE LOS DATOS CRUDOS ---
# Cargamos el dataset original de créditos (que tiene a los actores y directores)
df_credits = pd.read_csv('../data/raw/tmdb_5000_credits.csv')

# En tmdb_5000_credits, la llave se llama 'movie_id'. La renombramos a 'id' para que coincida
df_credits = df_credits.rename(columns={'movie_id': 'id'})

# Hacemos el Merge para traernos SOLO la columna 'crew' a tu tabla
if 'crew' not in df_transformado.columns:
    df_transformado = pd.merge(df_transformado, df_credits[['id', 'crew']], on='id', how='left')

# Usamos la función de Gene para extraer el nombre
df_transformado['director'] = df_transformado['crew'].apply(extraer_director)
df_transformado['director'] = df_transformado['director'].fillna('Desconocido')


# --- 2. ESTUDIOS PRODUCTORES ---
df_transformado['estudios_lista'] = df_transformado['production_companies'].apply(limpiar_estudios)

# Encoding de los 3 estudios más gigantes de Hollywood
df_transformado['es_warner'] = df_transformado['estudios_lista'].apply(lambda x: 1 if 'Warner Bros.' in x else 0)
df_transformado['es_universal'] = df_transformado['estudios_lista'].apply(lambda x: 1 if 'Universal Pictures' in x else 0)
df_transformado['es_paramount'] = df_transformado['estudios_lista'].apply(lambda x: 1 if 'Paramount Pictures' in x else 0)


# --- 3. EXPERIENCIA DEL DIRECTOR (El peso) ---
# Reemplazamos el nombre por la cantidad de películas que ha dirigido
frecuencia_directores = df_transformado['director'].value_counts()
df_transformado['experiencia_director'] = df_transformado['director'].map(frecuencia_directores)

print("Director recuperado desde los datos originales y nuevas variables creadas con éxito.")

Director recuperado desde los datos originales y nuevas variables creadas con éxito.


Calculamos el Retorno de Inversión ($ROI = \frac{Revenue - Budget}{Budget}$) operando columnas completas a la vez, lo cual es exponencialmente más rápido que usar un bucle for. Finalmente, reducimos el consumo de memoria RAM cambiando los tipos de datos de texto repetitivo a category.

In [ ]:
# Cálculo Vectorizado / Broadcasting para el ROI
df_transformado['roi'] = (df_transformado['revenue'] - df_transformado['budget']) / df_transformado['budget']

# Optimización de Memoria (Chunking conceptual de tipos de datos)
# Convertir variables de texto con pocas opciones únicas a tipo 'category'
columnas_a_optimizar = ['original_language', 'status', 'temporada']
for col in columnas_a_optimizar:
    df_transformado[col] = df_transformado[col].astype('category')

# Mostrar el ahorro de memoria y los nuevos datos
print("Memoria optimizada y ROI calculado.")
df_transformado[['title', 'release_date', 'director', 'roi', 'esrama_d']].head()

Memoria optimizada y ROI calculado.


,title,release_date,director,roi,es_drama
0,Avatar,2009-12-10,James Cameron,10.763566,0
1,Pirates of the Caribbean: At World's End,2007-05-19,Gore Verbinski,2.203333,0
2,Spectre,2015-10-26,Sam Mendes,2.594590,0
3,The Dark Knight Rises,2012-07-16,Christopher Nolan,3.339756,1
4,John Carter,2012-03-07,Andrew Stanton,0.092843,0


In [17]:
# Guardado del Dataset Final

ruta_final = '../data/processed/movies_listo_para_modelo.csv'
df_transformado.to_csv(ruta_final, index=False)

print(f"Dataset final guardado en: {ruta_final}")

Dataset final guardado en: ../data/processed/movies_listo_para_modelo.csv
